# Modelos ARIMA y SARIMA

La selección combina evidencia de ACF y PACF, una rejilla acotada y diagnóstico de residuos. AIC y BIC se comparan únicamente entre modelos ARIMA ajustados a la misma serie.

## Parámetros iniciales

En total, vía aérea y vía terrestre, la ACF en niveles decae lentamente y la PACF concentra señal en los primeros rezagos, por lo que se exploran p y q entre 0 y 2, con d=1. Los picos anuales justifican D=1 y componentes P y Q entre 0 y 1. El mismo rango se usa para El Salvador, Estados Unidos y Honduras, cuyas ACF diferenciadas conservan dependencia corta y señal en el rezago 12. Marítima requiere d=2 como escenario exploratorio porque ninguna transformación probada rechaza raíz unitaria; D=1 se conserva para capturar el ciclo anual, sin presentar la serie como estacionaria. La rejilla permite que AIC decida entre términos autorregresivos y de media móvil dentro de esos rangos, mientras Ljung-Box comprueba si queda autocorrelación residual.

In [1]:
from pathlib import Path
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

RAIZ = Path.cwd()
if not (RAIZ / "src").exists():
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ))

from src.modelos import ajustar_sarima, diagnostico_residuos, grid_sarima
from src.utils import RUTA_FIGURAS, RUTA_RESULTADOS, SERIES, cargar_serie

PARAMETROS = {
    clave: {"d": 2 if clave == "via_maritima" else 1, "D": 1}
    for clave in SERIES
}
RUTA_RESULTADOS.mkdir(parents=True, exist_ok=True)

In [2]:
def ejecutar_auto_arima(serie_log, d, D):
    try:
        from pmdarima import auto_arima

        modelo = auto_arima(
            serie_log,
            seasonal=True,
            m=12,
            d=d,
            D=D,
            start_p=0,
            start_q=0,
            max_p=2,
            max_q=2,
            start_P=0,
            start_Q=0,
            max_P=1,
            max_Q=1,
            stepwise=True,
            suppress_warnings=True,
            error_action="ignore",
        )
        return modelo.order, modelo.seasonal_order, modelo.aic()
    except Exception as error:
        return None, None, f"No disponible: {error}"


comparaciones = []
automaticos = []
for clave in SERIES:
    serie_log = np.log1p(cargar_serie(clave, "train"))
    parametros = PARAMETROS[clave]
    rejilla = grid_sarima(serie_log, **parametros)
    validos = rejilla[np.isfinite(rejilla["aic"])]
    convergentes = validos[validos["converged"]]
    candidatos = (convergentes if len(convergentes) >= 3 else validos).head(3)

    print(f"\n{SERIES[clave]}: top 5 por AIC")
    print(validos.head(5).to_string(index=False))

    auto_order, auto_seasonal, auto_aic = ejecutar_auto_arima(
        serie_log,
        parametros["d"],
        parametros["D"],
    )
    automaticos.append(
        {
            "serie": clave,
            "order": auto_order,
            "seasonal_order": auto_seasonal,
            "aic": auto_aic,
        }
    )
    print(
        "auto_arima:",
        auto_order,
        auto_seasonal,
        auto_aic,
    )

    for posicion, fila in candidatos.reset_index(drop=True).iterrows():
        modelo = ajustar_sarima(
            serie_log,
            tuple(fila["order"]),
            tuple(fila["seasonal_order"]),
        )
        residuos, figura = diagnostico_residuos(modelo)
        seleccionado = posicion == 0
        if seleccionado:
            ruta = RUTA_FIGURAS / f"modelo_{clave}_residuos.png"
            figura.savefig(ruta, dpi=150, bbox_inches="tight")
        plt.close(figura)
        comparaciones.append(
            {
                "serie": clave,
                "order": tuple(fila["order"]),
                "seasonal_order": tuple(fila["seasonal_order"]),
                "aic": modelo.aic,
                "bic": modelo.bic,
                "ljung_box_p": residuos["ljung_box_p"],
                "jarque_bera_p": residuos["jarque_bera_p"],
                "seleccionado": seleccionado,
            }
        )

modelos_arima = pd.DataFrame(comparaciones)
modelos_arima.to_csv(RUTA_RESULTADOS / "modelos_arima.csv", index=False)
auto_arima_resultados = pd.DataFrame(automaticos)
print("\nComparación manual consolidada")
print(modelos_arima.to_string(index=False))
print("\nResultados de auto_arima")
print(auto_arima_resultados.to_string(index=False))


Total: top 5 por AIC
    order seasonal_order       aic       bic  converged
(2, 1, 2)  (1, 1, 1, 12) 67.955300 87.409164      False
(1, 1, 2)  (1, 1, 1, 12) 68.547485 85.222226       True
(2, 1, 2)  (1, 1, 0, 12) 68.962388 85.687339      False
(2, 1, 2)  (0, 1, 1, 12) 69.193863 85.868604       True
(1, 1, 2)  (0, 1, 1, 12) 69.447859 83.343477       True


auto_arima: (0, 1, 0) (0, 1, 1, 12) 73.96452580289909



Vía Aérea: top 5 por AIC
    order seasonal_order        aic        bic  converged
(2, 1, 2)  (1, 1, 0, 12) 165.055365 181.780315       True
(2, 1, 2)  (1, 1, 1, 12) 165.206698 184.660562       True
(0, 1, 1)  (1, 1, 0, 12) 166.805096 175.217159       True
(1, 1, 0)  (1, 1, 0, 12) 167.327686 175.715058       True
(1, 1, 1)  (1, 1, 0, 12) 167.387581 178.570743       True


auto_arima: (0, 1, 1) (1, 1, 0, 12) 184.80154618984164



Vía Terrestre: top 5 por AIC
    order seasonal_order        aic        bic  converged
(0, 1, 1)  (1, 1, 0, 12) 124.179461 132.591524       True
(1, 1, 0)  (1, 1, 0, 12) 124.315480 132.702851       True
(2, 1, 2)  (0, 1, 1, 12) 124.942249 141.616990       True
(1, 1, 2)  (0, 1, 1, 12) 125.267318 139.162936       True
(1, 1, 2)  (1, 1, 0, 12) 125.313946 139.292899       True


...


Estados Unidos: top 5 por AIC
    order seasonal_order        aic        bic  converged
(1, 1, 2)  (1, 1, 1, 12) 365.515169 382.189910       True
(1, 1, 2)  (0, 1, 1, 12) 367.135371 381.030989      False
(2, 1, 1)  (1, 1, 0, 12) 368.343930 382.281388       True
(1, 1, 2)  (1, 1, 0, 12) 368.956487 382.935440       True
(2, 1, 2)  (1, 1, 0, 12) 369.366134 386.091085      False


auto_arima: (0, 1, 0) (0, 1, 0, 12) 404.6826726949767



Honduras: top 5 por AIC
    order seasonal_order        aic        bic  converged
(1, 1, 2)  (1, 1, 1, 12) 357.214088 373.888829      False
(2, 1, 2)  (1, 1, 0, 12) 359.014701 375.739651      False
(2, 1, 2)  (1, 1, 1, 12) 360.182134 379.635999      False
(1, 1, 2)  (0, 1, 1, 12) 360.205135 374.100753       True
(2, 1, 2)  (0, 1, 1, 12) 361.974603 378.649344      False


auto_arima: (2, 1, 1) (0, 1, 1, 12) 395.1913944523051



Comparación manual consolidada
              serie     order seasonal_order        aic        bic  ljung_box_p  jarque_bera_p  seleccionado
              total (1, 1, 2)  (1, 1, 1, 12)  68.547485  85.222226 4.570340e-03   0.000000e+00          True
              total (2, 1, 2)  (0, 1, 1, 12)  69.193863  85.868604 7.128616e-07   0.000000e+00         False
              total (1, 1, 2)  (0, 1, 1, 12)  69.447859  83.343477 8.796281e-06   0.000000e+00         False
          via_aerea (2, 1, 2)  (1, 1, 0, 12) 165.055365 181.780315 1.611343e-01   0.000000e+00          True
          via_aerea (2, 1, 2)  (1, 1, 1, 12) 165.206698 184.660562 2.248197e-01   0.000000e+00         False
          via_aerea (0, 1, 1)  (1, 1, 0, 12) 166.805096 175.217159 1.000000e+00   0.000000e+00         False
      via_terrestre (0, 1, 1)  (1, 1, 0, 12) 124.179461 132.591524 5.395256e-01   0.000000e+00          True
      via_terrestre (1, 1, 0)  (1, 1, 0, 12) 124.315480 132.702851 8.680528e-01   0.000000e+00  

## Criterio de selección

Para cada serie se retienen tres especificaciones convergentes con menor AIC y se comparan también por BIC, Ljung-Box y Jarque-Bera. `auto_arima` se restringe al mismo espacio de búsqueda, así que una propuesta con términos de orden bajo y diferenciación fijada es coherente con las ACF y PACF observadas. Si difiere del mínimo manual, la causa puede ser su búsqueda escalonada y no una contradicción con el diagnóstico. Si la biblioteca no carga por incompatibilidad binaria, la salida de la celda registra el error y la rejilla manual queda como método reproducible. Jarque-Bera se informa como diagnóstico, pero la falta de normalidad no invalida por sí sola el pronóstico; la ausencia de autocorrelación residual tiene mayor peso.

## Modelos alternativos

Se generan cuatro referencias con el mismo horizonte del test. Holt-Winters y suavizamiento exponencial simple se ajustan en `log1p` y se revierten a viajeros; seasonal naive trabaja en niveles y repite los últimos doce meses; Prophet usa estacionalidad anual sobre `log1p`. Si Prophet no está disponible, el error queda registrado y los otros tres modelos continúan.

In [3]:
from src.modelos import (
    modelo_holt_winters,
    modelo_prophet,
    modelo_seasonal_naive,
    modelo_ses,
)

RUTA_PREDICCIONES = RUTA_RESULTADOS / "predicciones"
RUTA_PREDICCIONES.mkdir(parents=True, exist_ok=True)
MODELOS_ALTERNATIVOS = {
    "holt_winters": modelo_holt_winters,
    "ses": modelo_ses,
    "seasonal_naive": modelo_seasonal_naive,
    "prophet": modelo_prophet,
}

def guardar_prediccion(clave, modelo, prediccion):
    salida = pd.DataFrame(
        {
            "fecha": prediccion.index.strftime("%Y-%m-%d"),
            "prediccion": prediccion.to_numpy(dtype=float),
        }
    )
    salida.to_csv(RUTA_PREDICCIONES / f"{clave}_{modelo}.csv", index=False)


for clave in SERIES:
    train = cargar_serie(clave, "train")
    horizonte = len(cargar_serie(clave, "test"))
    for nombre, funcion in MODELOS_ALTERNATIVOS.items():
        try:
            prediccion = funcion(train, horizonte)
            guardar_prediccion(clave, nombre, prediccion)
            print(clave, nombre, len(prediccion))
        except Exception as error:
            print(clave, nombre, f"no disponible: {error}")

total holt_winters 63
total ses 63
total seasonal_naive 63


Importing plotly failed. Interactive plots will not work.


15:32:25 - cmdstanpy - INFO - Chain [1] start processing


15:32:25 - cmdstanpy - INFO - Chain [1] done processing


15:32:25 - cmdstanpy - INFO - Chain [1] start processing


...

15:32:26 - cmdstanpy - INFO - Chain [1] done processing


pais_el_salvador ses 63
pais_el_salvador seasonal_naive 63
pais_el_salvador prophet 63
pais_estados_unidos holt_winters 63
pais_estados_unidos ses 63
pais_estados_unidos seasonal_naive 63
pais_estados_unidos prophet 63


15:32:26 - cmdstanpy - INFO - Chain [1] start processing


15:32:26 - cmdstanpy - INFO - Chain [1] done processing


pais_honduras holt_winters 63
pais_honduras ses 63
pais_honduras seasonal_naive 63
pais_honduras prophet 63
